**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Step 0 - import libraries needed for this lab
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    GradientBoostingClassifier, AdaBoostClassifier
)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Standard seed used throughout the notebook, for reproducibility
SEED = 42

In [2]:
# # Load the dataset
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


### Recreating the feature engineering from the previous lab

First we repeat the same cleaning/engineering steps used in the Feature Engineering lab, so the comparison between models is fair:
1. Drop rows with missing values.
2. Reduce `Cabin` to just the deck letter.
3. Drop `PassengerId` and `Name` (no predictive value).
4. One-hot encode the remaining categorical columns.

In [3]:
# 1.1 Same feature engineering as the previous lab
spaceship = spaceship.dropna()
spaceship["Cabin"] = spaceship["Cabin"].apply(lambda x: x.split("/")[0])
spaceship = spaceship.drop(columns=["PassengerId", "Name"])
spaceship = pd.get_dummies(spaceship, drop_first=True)

X = spaceship.drop(columns=["Transported"])
y = spaceship["Transported"]

X.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,False,False,False,False,False,False,False,True,False,False,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,True,False,False,False,False,False,False,False,False,False,False,True,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,True,False,False,False,False,False,False,False,False,False,False,True,False
4,16.0,303.0,70.0,151.0,565.0,2.0,False,False,False,False,False,False,False,True,False,False,False,True,False


### 1.2 Feature Scaling

Even though tree-based ensembles don't strictly need scaling, `StandardScaler` standardizes every feature to mean 0 / std 1, which keeps the comparison consistent and doesn't hurt these models.

In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)
X_scaled.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,0.695413,-0.345756,-0.285355,-0.309494,-0.273759,-0.269534,1.717147,-0.510811,-0.738664,3.085305,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
1,-0.336769,-0.176748,-0.279993,-0.266112,0.206165,-0.230494,-0.582361,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,1.438646,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
2,2.002842,-0.279083,1.845163,-0.309494,5.596357,-0.226058,1.717147,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,6.306963
3,0.282540,-0.345756,0.479034,0.334285,2.636384,-0.098291,1.717147,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
4,-0.887266,0.124056,-0.243650,-0.047470,0.220152,-0.267759,-0.582361,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,1.438646,-0.652578,-0.017402,-0.322689,0.666047,-0.158555


### 1.3 Feature Selection

We use `SelectKBest` with the ANOVA F-test (`f_classif`) to keep only the **10** features most statistically associated with `Transported`, dropping the weaker ones.

In [5]:
selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X_scaled, y)

selected_columns = X.columns[selector.get_support()]
X_selected = pd.DataFrame(X_selected, columns=selected_columns, index=X.index)

print("Selected features:", list(selected_columns))
X_selected.head()

Selected features: ['RoomService', 'Spa', 'VRDeck', 'HomePlanet_Europa', 'CryoSleep_True', 'Cabin_B', 'Cabin_C', 'Cabin_E', 'Cabin_F', 'Destination_TRAPPIST-1e']


,RoomService,Spa,VRDeck,HomePlanet_Europa,CryoSleep_True,Cabin_B,Cabin_C,Cabin_E,Cabin_F,Destination_TRAPPIST-1e
0,-0.345756,-0.273759,-0.269534,1.717147,-0.738664,3.085305,-0.312289,-0.339578,-0.695098,0.666047
1,-0.176748,0.206165,-0.230494,-0.582361,-0.738664,-0.324117,-0.312289,-0.339578,1.438646,0.666047
2,-0.279083,5.596357,-0.226058,1.717147,-0.738664,-0.324117,-0.312289,-0.339578,-0.695098,0.666047
3,-0.345756,2.636384,-0.098291,1.717147,-0.738664,-0.324117,-0.312289,-0.339578,-0.695098,0.666047
4,0.124056,0.220152,-0.267759,-0.582361,-0.738664,-0.324117,-0.312289,-0.339578,1.438646,0.666047


**2. Perform Train Test Split**

I split the scaled, feature-selected data into an 80/20 train/test set.

In [6]:
# 2.1 Train Test/Split
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=SEED
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (5284, 10)
X_test shape: (1322, 10)


**3. Model Selection** - now you will try to apply different ensemble methods in order to get a better model

We'll reuse a small helper to fit and evaluate each ensemble model consistently.

In [7]:
# 3.1 Model Selection
def evaluate(model, X_train, y_train, X_test, y_test, label="Model"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"--- {label} ---")
    print("Accuracy:", acc)
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    return acc

**3.2 Bagging and Pasting**

**Bagging** trains many Decision Trees, each on a random *bootstrap sample* (drawn with replacement) of the training data, then averages/votes their predictions. **Pasting** is the same idea but samples *without* replacement (`bootstrap=False`). I try Bagging here.

In [8]:
# 3.2 Bagging
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    bootstrap=True,   # True = Bagging, False = Pasting
    random_state=SEED,
    n_jobs=-1
)
acc_bagging = evaluate(bagging, X_train, y_train, X_test, y_test, "Bagging")

--- Bagging ---
Accuracy: 0.7692889561270801

Classification Report:
               precision    recall  f1-score   support

       False       0.79      0.73      0.76       653
        True       0.75      0.81      0.78       669

    accuracy                           0.77      1322
   macro avg       0.77      0.77      0.77      1322
weighted avg       0.77      0.77      0.77      1322

Confusion Matrix:
 [[475 178]
 [127 542]]


**3.3 Random Forests**

**Random Forest** is essentially Bagging with Decision Trees, plus an extra twist: each split only considers a random subset of features, which decorrelates the trees further and usually improves generalization.

In [9]:
# 3.3 Random Forest
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=SEED,
    n_jobs=-1
)
acc_rf = evaluate(random_forest, X_train, y_train, X_test, y_test, "Random Forest")

--- Random Forest ---
Accuracy: 0.7813918305597579

Classification Report:
               precision    recall  f1-score   support

       False       0.80      0.74      0.77       653
        True       0.76      0.82      0.79       669

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322

Confusion Matrix:
 [[484 169]
 [120 549]]


**3.4 Gradient Boosting**

**Gradient Boosting** builds trees *sequentially*, where each new tree is trained to correct the errors (residuals) of the previous ones, rather than training trees independently in parallel like bagging/random forests.

In [10]:
# 3.4 Gradient Boosting
gradient_boosting = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=SEED
)
acc_gb = evaluate(gradient_boosting, X_train, y_train, X_test, y_test, "Gradient Boosting")

--- Gradient Boosting ---
Accuracy: 0.783661119515885

Classification Report:
               precision    recall  f1-score   support

       False       0.83      0.71      0.76       653
        True       0.75      0.85      0.80       669

    accuracy                           0.78      1322
   macro avg       0.79      0.78      0.78      1322
weighted avg       0.79      0.78      0.78      1322

Confusion Matrix:
 [[465 188]
 [ 98 571]]


**3.5 Adaptive Boosting**

**AdaBoost (Adaptive Boosting)** also builds trees sequentially, but instead of fitting residuals directly, it re-weights the training examples after each round — misclassified points get more weight so the next weak learner focuses on them.

In [11]:
# 3.5 Adaptive Boosting
adaboost = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)
acc_ada = evaluate(adaboost, X_train, y_train, X_test, y_test, "AdaBoost")

--- AdaBoost ---
Accuracy: 0.7488653555219364

Classification Report:
               precision    recall  f1-score   support

       False       0.70      0.85      0.77       653
        True       0.82      0.65      0.72       669

    accuracy                           0.75      1322
   macro avg       0.76      0.75      0.75      1322
weighted avg       0.76      0.75      0.75      1322

Confusion Matrix:
 [[558  95]
 [237 432]]


**4. Which model is the best and why?**

In [12]:
# 4.1 Model Comparison
results = pd.DataFrame({
    "Model": ["Bagging", "Random Forest", "Gradient Boosting", "AdaBoost"],
    "Accuracy": [acc_bagging, acc_rf, acc_gb, acc_ada]
}).sort_values("Accuracy", ascending=False).reset_index(drop=True)

results

,Model,Accuracy
0,Gradient Boosting,0.783661
1,Random Forest,0.781392
2,Bagging,0.769289
3,AdaBoost,0.748865


In [13]:
# 4.2 Identifying the Best Model 
best_model = results.iloc[0]
print(f"Best model: {best_model['Model']} with accuracy {best_model['Accuracy']:.4f}")

Best model: Gradient Boosting with accuracy 0.7837


** All four ensemble methods beat the single-KNN baseline from the earlier labs (**77–79%).** On this run, **Gradient Boosting performed best (78.4% accuracy)**, narrowly ahead of **Random Forest (~78.1%)** and **Bagging (76.9%)**, with **AdaBoost trailing (74.9%)**.

- Gradient Boosting's edge makes sense: building trees sequentially to correct prior errors lets it fit the data's patterns more closely than the purely parallel, randomized Bagging/Random Forest approaches.
- Random Forest slightly outperforming plain Bagging fits expectations too — its extra randomness (random feature subsets per split) decorrelates the trees a bit more.
- AdaBoost coming in last here is a reminder that boosting isn't automatically better: it's more sensitive to hyperparameters (`n_estimators`, `learning_rate`) and to noisy/mislabeled points (since it up-weights hard examples), so with default-ish settings it can underperform the more robust Bagging/Random Forest on some datasets.

In short: **Gradient Boosting is the best model on this run**, but the gap to Random Forest is small, and results could shift with hyperparameter tuning (e.g. grid search on `n_estimators`, `max_depth`, `learning_rate`).